# 11. Datasets, supervision, and train/validation/test splits

**Level:** Beginner to intermediate

**Prerequisites:** 01, 07-08: learning terminology and data containers.

[Course index](00_START_HERE.ipynb) | [Previous](10_Math_Probability_and_Optimization.ipynb) | [Next](12_Preprocessing_and_Leakage_Safe_Pipelines.ipynb)

**Topics:** Classification vs regression, Supervised learning, Unsupervised learning, Reinforcement learning, Train data, Validation data, Test data, Leakage, Self-supervised learning, Semi-supervised learning.

**Matching script:** `examples\11_Datasets_Supervision_and_Data_Splits.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Supervised learning uses examples with target labels: regression predicts a quantity and
classification predicts a category or category probability. Unsupervised learning discovers
structure without those task labels, such as clusters or compressed representations.
Reinforcement learning selects actions whose consequences affect future observations and rewards.

Self-supervised learning constructs targets from the data itself, for example predicting the next
token. It is not 'no objective' learning. Semi-supervised learning combines labeled and unlabeled
examples. Transfer learning reuses a learned representation for a different task or domain.

Train data fits parameters; validation data selects hyperparameters and thresholds; test data
estimates performance once choices are fixed. Repeatedly checking test scores and selecting the
best variant turns that test set into validation data. Stratification preserves class proportions
approximately, not independence.

Split by time for future prediction, by device/customer for unseen-entity evaluation, or by
groups when repeated measurements would leak identity. Random row splits are suitable only when
their independence assumptions match deployment. A label collected after an outage must not
leak into features supposedly available before the outage.

## Visual guide

![Datasets, supervision, and train/validation/test splits teaching diagram](images/lifecycle.png)

Trace the diagram left to right and identify the input, transformation, and output. It illustrates the mechanism, not measured model quality.

## Mathematics or data contract

A 60/20/20 split allocates 60% to fitting, 20% to model selection, and 20% to final evaluation.

## Important limitation

This example uses independent synthetic rows; real telemetry usually needs chronological or grouped splits.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Disjoint stratified data partitions

**Purpose:** Track sample IDs to prove split isolation.

### Read the code in this order
1. Generate labeled synthetic samples with a fixed seed.
2. Reserve test data first.
3. Split the remaining development data into train and validation.
4. Assert disjoint IDs and full coverage.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
train/validation/test: 120 40 40
feature shape: (200, 6) target shape: (200,)
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=200, n_features=6, random_state=42)
ids = np.arange(len(y))
dev, test = train_test_split(ids, test_size=0.2, stratify=y, random_state=42)
train, validation = train_test_split(dev, test_size=0.25, stratify=y[dev], random_state=42)
assert [len(train), len(validation), len(test)] == [120, 40, 40]
assert not (set(train) & set(validation) or set(train) & set(test) or set(validation) & set(test))
assert set(train) | set(validation) | set(test) == set(ids)
print("train/validation/test:", len(train), len(validation), len(test))
print("feature shape:", X.shape, "target shape:", y.shape)

train/validation/test: 120 40 40
feature shape: (200, 6) target shape: (200,)


### Interpret example 1

The result should demonstrate this purpose: **Track sample IDs to prove split isolation.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Replace the row split with GroupShuffleSplit keyed by device and prove no device occurs in both sets.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Classification vs regression` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A 60/20/20 split allocates 60% to fitting, 20% to model selection, and 20% to final evaluation.
- Name the boundary of the demonstration: This example uses independent synthetic rows; real telemetry usually needs chronological or grouped splits.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.